In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd /content/smallnet

!git fetch origin
!git pull --ff-only origin main

/content/smallnet
From https://github.com/SepehrAkbari/smallnet
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
import json

with open("configs/camvid_vgg_cp_paper.json") as f:
    config = json.load(f)

print(config["cp_iteration_sensitivity"]["iteration_budgets"])

[10, 25, 50, 100, 200, 400]


In [4]:
!uv sync
!uv run python -m pytest -q

Resolved 85 packages in 1ms
Checked 79 packages in 1ms
.....................................s................................   [100%]
69 passed, 1 skipped in 26.88s


In [5]:
import pandas as pd
from pathlib import Path

summary_path = Path(
    "results/camvid_vgg_cp/cp_iteration_sensitivity_summary.csv"
)

assert summary_path.exists(), "Existing sensitivity summary is missing."

sensitivity = pd.read_csv(summary_path)

for column in ["rank", "seed", "iteration_budget"]:
    sensitivity[column] = pd.to_numeric(
        sensitivity[column],
        errors="coerce",
    )

completed = sensitivity[
    sensitivity["status"] == "completed"
]

print("Completed rows:", len(completed))
print(
    completed.groupby(
        ["rank", "iteration_budget"]
    )["seed"].nunique()
)

Completed rows: 36
rank  iteration_budget
128   10                  3
      25                  3
      50                  3
      100                 3
256   10                  3
      25                  3
      50                  3
      100                 3
512   10                  3
      25                  3
      50                  3
      100                 3
Name: seed, dtype: int64


In [6]:
!mkdir -p /content/smallnet/model

!rsync -a \
  /content/drive/MyDrive/model/ \
  /content/smallnet/model/
  
!ls -lh model/best_model.pth
!sha256sum model/best_model.pth

-rw------- 1 root root 513M May 21 16:33 model/best_model.pth
1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3  model/best_model.pth


In [7]:
!ls -lh model/best_model.pth
!nvidia-smi

-rw------- 1 root root 513M May 21 16:33 model/best_model.pth
Wed Jul 22 20:01:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |          

In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None",
)

CUDA available: True
GPU: Tesla T4


In [9]:
import subprocess
from pathlib import Path

REPO = Path("/content/smallnet")
BACKUP = Path(
    "/content/drive/MyDrive/smallnet_colab_backup"
)

def backup_results():
    (BACKUP / "camvid_vgg_cp").mkdir(
        parents=True,
        exist_ok=True,
    )
    (BACKUP / "paper").mkdir(
        parents=True,
        exist_ok=True,
    )

    subprocess.run(
        [
            "rsync",
            "-a",
            f"{REPO}/results/camvid_vgg_cp/",
            f"{BACKUP}/camvid_vgg_cp/",
        ],
        check=True,
    )

    subprocess.run(
        [
            "rsync",
            "-a",
            f"{REPO}/results/paper/",
            f"{BACKUP}/paper/",
        ],
        check=True,
    )

    print("Backup complete.")

In [10]:
backup_results()

Backup complete.


In [11]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 128 \
  --seeds 0 1 2 \
  --iteration-budgets 200

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [12]:
backup_results()

Backup complete.


In [13]:
sensitivity = pd.read_csv(summary_path)
print("Rows:", len(sensitivity))

Rows: 39


In [14]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 128 \
  --seeds 0 1 2 \
  --iteration-budgets 400

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [15]:
backup_results()

Backup complete.


In [16]:
print(
    len(
        pd.read_csv(summary_path)
    )
)

42


In [17]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 256 \
  --seeds 0 1 2 \
  --iteration-budgets 200

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [18]:
backup_results()

Backup complete.


In [19]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 256 \
  --seeds 0 1 2 \
  --iteration-budgets 400

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [20]:
backup_results()

Backup complete.


In [21]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2 \
  --iteration-budgets 200

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [22]:
backup_results()

Backup complete.


In [23]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage cp-iteration-sensitivity \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2 \
  --iteration-budgets 400

Wrote:
  /content/smallnet/results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json


In [24]:
backup_results()

Backup complete.


In [25]:
print(
    len(
        pd.read_csv(summary_path)
    )
)

54


In [29]:
import csv
import json
from pathlib import Path

root = Path("results/camvid_vgg_cp")

with (
    root / "cp_iteration_sensitivity_summary.csv"
).open(newline="") as f:
    rows = list(csv.DictReader(f))

with (
    root
    / "cp_iteration_sensitivity_budget_transitions.csv"
).open(newline="") as f:
    transitions = list(csv.DictReader(f))

metadata = json.loads(
    (
        root
        / "cp_iteration_sensitivity_metadata.json"
    ).read_text()
)

assert len(rows) == 54
assert sum(
    row["status"] == "completed"
    for row in rows
) == 54

assert len(
    {
        (
            row["rank"],
            row["seed"],
            row["iteration_budget"],
        )
        for row in rows
    }
) == 54

assert {
    int(row["iteration_budget"])
    for row in rows
} == {
    10,
    25,
    50,
    100,
    200,
    400,
}

assert len(transitions) == 15

assert {
    (
        int(row["lower_budget"]),
        int(row["upper_budget"]),
    )
    for row in transitions
} == {
    (10, 25),
    (25, 50),
    (50, 100),
    (100, 200),
    (200, 400),
}

for rank, seed in {
    (
        row["rank"],
        row["seed"],
    )
    for row in rows
}:
    group = [
        row
        for row in rows
        if row["rank"] == rank
        and row["seed"] == seed
    ]

    assert len(group) == 6

    assert len(
        {
            row[
                "initialization_hash_sha256"
            ]
            for row in group
        }
    ) == 1

    assert all(
        row[
            "initialization_hash_sha256"
        ]
        == row[
            "actual_fit_initialization_hash_sha256"
        ]
        for row in group
    )

assert metadata["failures"] == []
assert metadata[
    "figure_generation_failures"
] == []
assert metadata[
    "audit_generation_failures"
] == []
assert metadata["audit_complete"] is True

print(
    "Canonical six-budget sensitivity validation: PASS"
)

Canonical six-budget sensitivity validation: PASS


In [30]:
from pathlib import Path

audit = Path(
    "results/paper/"
    "cp_iteration_sensitivity_audit.md"
)

print(audit.read_text())

# CP iteration-budget sensitivity audit

Status: **complete**.

This diagnostic reports completed requested budgets and residual stabilization. It does not claim certified convergence because no per-iteration convergence history is available.

## Rank-level diagnostics

| Rank | Budget | Mean squared residual | Population SD | Seed range | Mean gap above bound |
|---:|---:|---:|---:|---:|---:|
| 128 | 10 | 0.904099241 | 0.000113080 | 0.000274071 | 0.171784210 |
| 128 | 25 | 0.898148846 | 0.000083929 | 0.000188989 | 0.165833816 |
| 128 | 50 | 0.895245000 | 0.000156815 | 0.000368376 | 0.162929970 |
| 128 | 100 | 0.893318771 | 0.000174707 | 0.000420873 | 0.161003741 |
| 128 | 200 | 0.891966190 | 0.000134121 | 0.000318529 | 0.159651160 |
| 128 | 400 | 0.891338914 | 0.000157598 | 0.000367949 | 0.159023884 |
| 256 | 10 | 0.864526949 | 0.000013443 | 0.000029993 | 0.204287830 |
| 256 | 25 | 0.855216880 | 0.000012177 | 0.000028228 | 0.194977761 |
| 256 | 50 | 0.850743058 | 0.000066848 | 0.00015

In [31]:
transitions = pd.read_csv(
    "results/camvid_vgg_cp/"
    "cp_iteration_sensitivity_budget_transitions.csv"
)

final_transition = transitions[
    (transitions["lower_budget"] == 200)
    & (transitions["upper_budget"] == 400)
]

display(
    final_transition.sort_values("rank")
)

,lower_actual_relative_squared_frobenius_error_seed_range,lower_budget,lower_mean_actual_relative_squared_frobenius_error,mean_absolute_change_below_1e_minus_3,mean_absolute_squared_residual_reduction,mean_relative_change_below_1_percent,mean_relative_squared_residual_reduction,method,rank,seed_range_change,upper_actual_relative_squared_frobenius_error_seed_range,upper_budget,upper_mean_actual_relative_squared_frobenius_error
4,0.000319,200,0.891966,True,0.000627,True,0.000703,cp,128,0.000049,0.000368,400,0.891339
9,0.000169,200,0.846052,True,0.000776,True,0.000917,cp,256,0.001197,0.001367,400,0.845277
14,0.000212,200,0.780846,False,-0.018974,False,-0.024299,cp,512,0.020469,0.020681,400,0.799820


In [33]:
candidate_columns = [
    column
    for column in final_transition.columns
    if "absolute" in column.lower()
    and "reduction" in column.lower()
]

print(candidate_columns)

print(final_transition.columns.tolist())
display(final_transition)

['mean_absolute_squared_residual_reduction']
['lower_actual_relative_squared_frobenius_error_seed_range', 'lower_budget', 'lower_mean_actual_relative_squared_frobenius_error', 'mean_absolute_change_below_1e_minus_3', 'mean_absolute_squared_residual_reduction', 'mean_relative_change_below_1_percent', 'mean_relative_squared_residual_reduction', 'method', 'rank', 'seed_range_change', 'upper_actual_relative_squared_frobenius_error_seed_range', 'upper_budget', 'upper_mean_actual_relative_squared_frobenius_error']


,lower_actual_relative_squared_frobenius_error_seed_range,lower_budget,lower_mean_actual_relative_squared_frobenius_error,mean_absolute_change_below_1e_minus_3,mean_absolute_squared_residual_reduction,mean_relative_change_below_1_percent,mean_relative_squared_residual_reduction,method,rank,seed_range_change,upper_actual_relative_squared_frobenius_error_seed_range,upper_budget,upper_mean_actual_relative_squared_frobenius_error
4,0.000319,200,0.891966,True,0.000627,True,0.000703,cp,128,0.000049,0.000368,400,0.891339
9,0.000169,200,0.846052,True,0.000776,True,0.000917,cp,256,0.001197,0.001367,400,0.845277
14,0.000212,200,0.780846,False,-0.018974,False,-0.024299,cp,512,0.020469,0.020681,400,0.799820


In [34]:
!ls -lh \
  results/paper/figures/cp_iteration_sensitivity.csv \
  results/paper/figures/cp_iteration_sensitivity.pdf \
  results/paper/figures/cp_iteration_sensitivity.png \
  results/paper/cp_iteration_sensitivity_audit.md

-rw-r--r-- 1 root root 5.7K Jul 22 20:13 results/paper/cp_iteration_sensitivity_audit.md
-rw-r--r-- 1 root root 3.9K Jul 22 20:13 results/paper/figures/cp_iteration_sensitivity.csv
-rw-r--r-- 1 root root  23K Jul 22 20:13 results/paper/figures/cp_iteration_sensitivity.pdf
-rw-r--r-- 1 root root 195K Jul 22 20:13 results/paper/figures/cp_iteration_sensitivity.png


In [35]:
backup_results()

Backup complete.


In [36]:
!cd /content/smallnet && \
  zip -r \
  /content/smallnet_cp_iteration_sensitivity_6budget.zip \
  results/camvid_vgg_cp/cp_iteration_sensitivity_summary.csv \
  results/camvid_vgg_cp/cp_iteration_sensitivity_rank_summary.csv \
  results/camvid_vgg_cp/cp_iteration_sensitivity_budget_transitions.csv \
  results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json \
  results/camvid_vgg_cp/cp_iteration_sensitivity_config_used.json \
  results/paper/cp_iteration_sensitivity_audit.md \
  results/paper/figures/cp_iteration_sensitivity.csv \
  results/paper/figures/cp_iteration_sensitivity.pdf \
  results/paper/figures/cp_iteration_sensitivity.png

  adding: results/camvid_vgg_cp/cp_iteration_sensitivity_summary.csv (deflated 93%)
  adding: results/camvid_vgg_cp/cp_iteration_sensitivity_rank_summary.csv (deflated 79%)
  adding: results/camvid_vgg_cp/cp_iteration_sensitivity_budget_transitions.csv (deflated 62%)
  adding: results/camvid_vgg_cp/cp_iteration_sensitivity_metadata.json (deflated 74%)
  adding: results/camvid_vgg_cp/cp_iteration_sensitivity_config_used.json (deflated 68%)
  adding: results/paper/cp_iteration_sensitivity_audit.md (deflated 66%)
  adding: results/paper/figures/cp_iteration_sensitivity.csv (deflated 69%)
  adding: results/paper/figures/cp_iteration_sensitivity.pdf (deflated 38%)
  adding: results/paper/figures/cp_iteration_sensitivity.png (deflated 17%)


In [37]:
!cp \
  /content/smallnet_cp_iteration_sensitivity_6budget.zip \
  /content/drive/MyDrive/smallnet_colab_backup/

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
set -e

cd /content

if [ -d smallnet/.git ]; then
    cd smallnet
    git fetch origin
    git pull --ff-only origin main
else
    git clone https://github.com/SepehrAkbari/smallnet.git
fi

Cloning into 'smallnet'...
Updating files: 100% (1657/1657), done.


In [5]:
%cd /content/smallnet
!git pull --ff-only origin main
!uv sync
!uv run python -m pytest -q

/content/smallnet
From https://github.com/SepehrAkbari/smallnet
 * branch            main       -> FETCH_HEAD
Already up to date.
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 25ms
Prepared 79 packages in 1m 06s                                           
Installed 79 packages in 1.01s                              
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.3.1
 + executing==2.2.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jinja2==3.1.6
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + kiwisolver==1.5.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mpmath==1.3.0
 + nest-as

In [7]:
!mkdir -p /content/smallnet/model

!rsync -a \
  /content/drive/MyDrive/model/ \
  /content/smallnet/model/

In [8]:
!ls -lh model/best_model.pth
!sha256sum model/best_model.pth

-rw------- 1 root root 513M May 21 16:33 model/best_model.pth
1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3  model/best_model.pth


In [9]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage rank512-stability \
  --device cuda \
  --ranks 512 \
  --seeds 0 1 2 \
  --iteration-budgets 200 400 \
  --repetitions 0 1 \
  --optimization-precisions float32

Traceback (most recent call last):
  File "/content/smallnet/scripts/run_experiment.py", line 204, in <module>
    main()
  File "/content/smallnet/scripts/run_experiment.py", line 196, in main
    outputs.append(runner(config, ROOT, device, max_batches=args.max_batches))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/smallnet/src/smallnet/experiment.py", line 1992, in run_rank512_stability
    metadata_path = save_manifest(
                    ^^^^^^^^^^^^^^
  File "/content/smallnet/src/smallnet/results.py", line 65, in save_manifest
    "environment": environment_metadata(device=device),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/smallnet/src/smallnet/results.py", line 40, in environment_metadata
    metadata["device_name"] = device_name(device)
                              ^^^^^^^^^^^^^^^^^^^
  File "/content/smallnet/src/smallnet/reproducibility.py", line 50, in device_name
    return torch.cuda.get_devi

In [ ]:
!rsync -a \
  /content/smallnet/results/camvid_vgg_cp/ \
  /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp/

!rsync -a \
  /content/smallnet/results/paper/ \
  /content/drive/MyDrive/smallnet_colab_backup/paper/